# Analisis de producto con CrewAI

Tres agentes coordinados por CrewAI usando un Qwen2.5-0.5B-Instruct.

In [1]:
%%capture
!pip -q install "crewai==1.14.4" "transformers>=4.45.0" "torch"

In [2]:
import os
from typing import Any, Dict, List, Optional, Union

import torch
from crewai import Agent, BaseLLM, Crew, Process, Task
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as hf_logging

os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
hf_logging.set_verbosity_error()

os.environ.setdefault("HOME", os.getcwd())
os.environ.setdefault("CREWAI_STORAGE_DIR", "crewai_storage")
os.environ.setdefault("OTEL_SDK_DISABLED", "true")
os.environ.setdefault("CREWAI_DISABLE_TELEMETRY", "true")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"


class HuggingFaceCrewLLM(BaseLLM):
    def __init__(self, model_name: str = MODEL_NAME, max_new_tokens: int = 180):
        super().__init__(model=model_name, temperature=0)
        self.model_name = model_name
        self.max_new_tokens = max_new_tokens
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(model_name, local_files_only=True)
            self.model_obj = AutoModelForCausalLM.from_pretrained(model_name, local_files_only=True)
        except Exception:
            self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
            self.model_obj = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True)
        self.model_obj.eval()

    def call(
        self,
        messages: Union[str, List[Dict[str, str]]],
        tools: Optional[List[dict]] = None,
        callbacks: Optional[List[Any]] = None,
        available_functions: Optional[Dict[str, Any]] = None,
        **kwargs,
    ) -> str:
        if isinstance(messages, str):
            prompt = messages
        else:
            prompt = "\n".join(str(message.get("content", "")) for message in messages)

        messages_for_model = [
            {"role": "system", "content": "Eres un asistente técnico. Responde en español claro. No repitas instrucciones. Si el tema es RAG, recuerda que significa Retrieval-Augmented Generation."},
            {"role": "user", "content": prompt},
        ]
        prompt = self.tokenizer.apply_chat_template(messages_for_model, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
        with torch.no_grad():
            output = self.model_obj.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id,
            )
        generated = self.tokenizer.decode(
            output[0][inputs["input_ids"].shape[-1]:],
            skip_special_tokens=True,
        ).strip()
        if not generated:
            generated = "resultado generado por el modelo local."
        return "Thought: El modelo local genero una respuesta.\nFinal Answer: " + generated

    def supports_function_calling(self) -> bool:
        return False

    def supports_stop_words(self) -> bool:
        return False


llm_local = HuggingFaceCrewLLM()
print("Modelo real cargado:", llm_local.model_name)

/Users/marcechris/Documents/Codex/2026-06-22/a/work/notebook_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 13647.05it/s]

Modelo real cargado: Qwen/Qwen2.5-0.5B-Instruct


In [3]:
analista_producto = Agent(role="Analista de producto", goal="Proponer funcionalidades de IA para una app educativa.", backstory="Convierte necesidades de usuarios en funcionalidades concretas.", llm=llm_local, verbose=True, allow_delegation=False)
especialista_riesgos = Agent(role="Especialista de riesgos de IA", goal="Detectar riesgos de privacidad, sesgo y calidad.", backstory="Revisa soluciones de IA antes de lanzarlas.", llm=llm_local, verbose=True, allow_delegation=False)
coordinador = Agent(role="Coordinador ejecutivo", goal="Integrar producto y riesgos en una recomendacion accionable.", backstory="Sintetiza informacion de varios especialistas.", llm=llm_local, verbose=True, allow_delegation=False)

print("Agentes cargados:", len([analista_producto, especialista_riesgos, coordinador]))

Agentes cargados: 3


In [4]:
tarea_producto = Task(description="Propón funcionalidades concretas de IA para una app educativa de repaso. Entrega solo la propuesta.", expected_output="Funcionalidades con usuario objetivo y beneficio.", agent=analista_producto)
tarea_riesgos = Task(description="Evalua riesgos de privacidad, sesgo y calidad de las funcionalidades. Entrega riesgos y mitigaciones.", expected_output="Riesgos y mitigaciones.", agent=especialista_riesgos, context=[tarea_producto])
tarea_recomendacion = Task(description="Redacta una recomendacion ejecutiva breve integrando propuesta y riesgos. Cierra con una accion recomendada.", expected_output="Recomendacion ejecutiva breve.", agent=coordinador, context=[tarea_producto, tarea_riesgos])

crew = Crew(agents=[analista_producto, especialista_riesgos, coordinador], tasks=[tarea_producto, tarea_riesgos, tarea_recomendacion], process=Process.sequential, verbose=True)
resultado = crew.kickoff()
print("\nResultado final:")
print(resultado.raw)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.4                                                                                        │
│  Latest version:  1.14.7                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: bbdf1f7f-c064-4783-81eb-2e009de7b8d5                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Propón funcionalidades concretas de IA para una app educativa de repaso. Entrega solo la propuesta.      │
│  ID: a230beda-c437-4826-86cf-dbfc3bc62d01                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Analista de producto                                                                                    │
│                                                                                                                 │
│  Task: Propón funcionalidades concretas de IA para una app educativa de repaso. Entrega solo la propuesta.      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Analista de producto                                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Funcionalidades concretas de IA para una app educativa de repaso:                                              │
│                                                                                                                 │
│  1. **Recomendación de recursos**: Implementar algoritmos de recomendación basados en el historial de           │
│  aprendizaje del usuario para sugerirle contenido relevante para su estudio.                                    │
│                                                                                                                 │
│  2. **Análisis de habilidades**: Utilizar técnicas de aprendizaje automático para analizar el rendimiento del   │
│  usuario y proporcionar ajustes de nivel para mejorar sus habilidades.                                          │
│                                                                                                                 │
│  3. **Generación de ejercicios complejos**: Crear modelos de generación de ejercicios complejos basados en el   │
│  rendimiento del usuario para ayudarlo a desarrollar habilidades más avanzadas.                                 │
│                                                                                                                 │
│  4. **Personalización de contenido**: Implementar algoritmos de personalización para adaptarse al rendimiento   │
│  del usuario, ofreciendo contenido específico que sea relevante para su nivel de                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Coordinador ejecutivo                                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Para integrar productividad y riesgos en una recomendación ejecutiva, se recomienda implementar algoritmos de  │
│  recomendación basados en el historial de aprendizaje del usuario para sugerirle contenido relevante para su    │
│  estudio. Además, se debe considerar la personalización de contenido basada en el rendimiento del usuario para  │
│  mejorar las habilidades y adaptabilidad. Para mitigar riesgos como privacidad y sesgo, se deben implementar    │
│  medidas de acceso limitado y políticas de privacidad éticas.                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Redacta una recomendacion ejecutiva breve integrando propuesta y riesgos. Cierra con una accion          │
│  recomendada.                                                                                                   │
│  Agent: Coordinador ejecutivo                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: bbdf1f7f-c064-4783-81eb-2e009de7b8d5                                                                       │
│  Final Output: Para integrar productividad y riesgos en una recomendación ejecutiva, se recomienda implementar  │
│  algoritmos de recomendación basados en el historial de aprendizaje del usuario para sugerirle contenido        │
│  relevante para su estudio. Además, se debe considerar la personalización de contenido basada en el             │
│  rendimiento del usuario para mejorar las habilidades y adaptabilidad. Para mitigar riesgos como privacidad y   │
│  sesgo, se deben implementar medidas de acceso limitado y políticas de privacidad éticas.                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Resultado final:
Para integrar productividad y riesgos en una recomendación ejecutiva, se recomienda implementar algoritmos de recomendación basados en el historial de aprendizaje del usuario para sugerirle contenido relevante para su estudio. Además, se debe considerar la personalización de contenido basada en el rendimiento del usuario para mejorar las habilidades y adaptabilidad. Para mitigar riesgos como privacidad y sesgo, se deben implementar medidas de acceso limitado y políticas de privacidad éticas.


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯